In [1]:
import string
import numpy as np
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
from sentence_transformers import CrossEncoder

from langchain_community.document_loaders.csv_loader import CSVLoader

* Display First Document Content and Metadata

In [2]:
file_path = "../clean_data/employee_data.csv"

try:
    loader = CSVLoader(file_path=file_path)
    documents = loader.load()

    print(f"Loaded {len(documents)} documents \n")
    print('First Document Content:\n', documents[0].page_content, '\n')
    print('Document Metadata -->', documents[0].metadata)
except FileNotFoundError:
    print(f"Error: File '{file_path}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")


Loaded 1000 documents 

First Document Content:
 name: AARON, JEFFERY M
job_titles: LIEUTENANT
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 165624.0
typical_hours: 36.40625
hourly_rate: 44.2503125 

Document Metadata --> {'source': '../clean_data/employee_data.csv', 'row': 0}


In [3]:
print(type(documents))

<class 'list'>


#### Function to Load Documents

In [4]:
def load_data(file_path:str, i:int):
    try:
        loader = CSVLoader(file_path=file_path)
        documents = loader.load()
    
        print(f"Loaded the {i}th document out of all {len(documents)} documents \n")
        print('Document Content:\n', documents[i].page_content, '\n')
        print('Document Metadata -->', documents[i].metadata)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
    except Exception as e:
        print(f"An error occurred: {e}")


* Display Last Document Content and Metadata

In [5]:
docs_999 = load_data(file_path="../clean_data/employee_data.csv", i=999)
docs_999

Loaded the 999th document out of all 1000 documents 

Document Content:
 name: ARANDA, CLAUDIA
job_titles: INVESTIGATOR II - IG
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 101628.0
typical_hours: 36.40625
hourly_rate: 44.2503125 

Document Metadata --> {'source': '../clean_data/employee_data.csv', 'row': 999}


***Lexical Search Using BM25***

In [6]:
def bm25_tokenizer(doc):
    """Tokenize text for BM25 search"""
    tokenized_doc = []
    for token in doc.lower().split():
        token = token.strip(string.punctuation)
        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

In [7]:
# Extract the text content from Document objects
print("Extracting doc from documents...\n")
docs = [doc.page_content for doc in documents]

# Tokenize the corpus
print("Tokenizing corpus for BM25...")
tokenized_corpus = []
for doc in tqdm(docs, desc="Building BM25 index"):
    tokenized_corpus.append(bm25_tokenizer(doc))  

# Create BM25 index
bm25 = BM25Okapi(tokenized_corpus)
print(f"BM25 index ready with {len(tokenized_corpus)} documents")


Extracting doc from documents...

Tokenizing corpus for BM25...


Building BM25 index:   0%|          | 0/1000 [00:00<?, ?it/s]

BM25 index ready with 1000 documents


In [8]:
def keyword_search(query, top_k=3):
    """Search using BM25"""
    print(f"Searching for: '{query}'")
    print("="*80)
    
    # Get BM25 scores
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    
    # Get top results
    top_n = np.argpartition(bm25_scores, -top_k)[-top_k:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    # Display results
    print(f"\nTop {top_k} results:\n")
    for i, hit in enumerate(bm25_hits):
        print(f"{i+1}. Score: {hit['score']:.3f}")
        print(f"   Document ID: {hit['corpus_id']}")
        print(f"   Content:\n   {docs[hit['corpus_id']][:300]}")
        print()
    
    return bm25_hits

* Test Keyword search

In [9]:
query1 = input('Enter query here: ')
results1 = keyword_search(query1, top_k=3)

Enter query here:  INVESTIGATOR INSPECTOR GENERAL


Searching for: 'INVESTIGATOR INSPECTOR GENERAL'

Top 3 results:

1. Score: 13.529
   Document ID: 999
   Content:
   name: ARANDA, CLAUDIA
job_titles: INVESTIGATOR II - IG
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 101628.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. Score: 12.175
   Document ID: 35
   Content:
   name: ABRAHAM, MICHAEL J
job_titles: ASST INSPECTOR GENERAL
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 115992.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. Score: 8.463
   Document ID: 151
   Content:
   name: ADAMS, RAYMOND B
job_titles: INFORMATION ANALYST (IGO)
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 72996.0
typical_hours: 36.40625
hourly_rate: 44.2503125



In [10]:
query2 = input('Enter query here: ')
results2 = keyword_search(query2, top_k=3)

Enter query here:  Employees with lastname ARANDA


Searching for: 'Employees with lastname ARANDA'

Top 3 results:

1. Score: 6.048
   Document ID: 998
   Content:
   name: ARANDA, ARTHUR
job_titles: LIEUTENANT
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 152010.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. Score: 5.769
   Document ID: 997
   Content:
   name: ARANDA, ALFREDO
job_titles: POLICE OFFICER
department: CHICAGO POLICE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 111252.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. Score: 5.639
   Document ID: 999
   Content:
   name: ARANDA, CLAUDIA
job_titles: INVESTIGATOR II - IG
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 101628.0
typical_hours: 36.40625
hourly_rate: 44.2503125



***Lexical Search with Reranking***

In [11]:
# Initialize the reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [12]:
def keyword_search_with_reranking(query, top_k=3, num_candidates=10):
    """
    BM25 keyword search with CrossEncoder reranking.
    
    Args:
        query: Search query string
        top_k: Number of final results to return after reranking
        num_candidates: Number of candidates to retrieve with BM25 before reranking
    
    Returns:
        List of reranked results with scores
    """
    print(f"Searching for: '{query}'")
    print("="*80)
    
    # Step 1: BM25 retrieval
    print(f"\n Step 1: BM25 retrieval (getting {num_candidates} candidates)")
    print("-"*80)
    
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)
    
    print(f"\nTop-3 BM25 results (before reranking):")
    for i, hit in enumerate(bm25_hits[:3]):
        print(f"\n{i+1}. BM25 Score: {hit['score']:.3f}")
        print(f"   {docs[hit['corpus_id']][:300]}")
    
    # Step 2: Rerank
    print(f"\n Step 2: Reranking with CrossEncoder")
    print("-"*80)
    
    pairs = [(query, docs[hit['corpus_id']]) for hit in bm25_hits]
    rerank_scores = reranker.predict(pairs)
    
    # Create results with Document objects
    reranked_results = []
    for hit, rerank_score in zip(bm25_hits, rerank_scores):
        reranked_results.append({
            'document': documents[hit['corpus_id']],  
            'corpus_id': hit['corpus_id'],
            'bm25_score': hit['score'],
            'rerank_score': float(rerank_score)
        })
    
    # Sort by rerank score
    reranked_results = sorted(reranked_results, key=lambda x: x['rerank_score'], reverse=True)
    final_results = reranked_results[:top_k]
    
    # Display results
    print(f"\n Top-{top_k} results after reranking:\n")
    for i, result in enumerate(final_results):
        doc = result['document']
        print(f"{i+1}. Rerank Score: {result['rerank_score']:.4f} (BM25: {result['bm25_score']:.3f})")
        print(f"   Document ID: {result['corpus_id']}")
        print(f"   Metadata: {doc.metadata}")
        print(f"   Content:\n   {doc.page_content[:300]}")
        print()
    
    return final_results


In [15]:
query1 = input('Enter query here: ')
results1 = keyword_search_with_reranking(query1, top_k=3, num_candidates=3)

Enter query here:  INVESTIGATOR INSPECTOR GENERAL


Searching for: 'INVESTIGATOR INSPECTOR GENERAL'

 Step 1: BM25 retrieval (getting 3 candidates)
--------------------------------------------------------------------------------

Top-3 BM25 results (before reranking):

1. BM25 Score: 13.529
   name: ARANDA, CLAUDIA
job_titles: INVESTIGATOR II - IG
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 101628.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 12.175
   name: ABRAHAM, MICHAEL J
job_titles: ASST INSPECTOR GENERAL
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 115992.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 8.463
   name: ADAMS, RAYMOND B
job_titles: INFORMATION ANALYST (IGO)
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 72996.0
typical_hours: 36.40625
hourly_rate: 44.2503125

 Step 2: Reranking with CrossEncoder
-------------------

In [17]:
query2 = input('Enter query here: ')
results2 = keyword_search_with_reranking(query2, top_k=3, num_candidates=3)

Enter query here:  General Labourers


Searching for: 'General Labourers'

 Step 1: BM25 retrieval (getting 3 candidates)
--------------------------------------------------------------------------------

Top-3 BM25 results (before reranking):

1. BM25 Score: 5.923
   name: ABRAHAM, MICHAEL J
job_titles: ASST INSPECTOR GENERAL
department: OFFICE OF INSPECTOR GENERAL
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 115992.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 4.210
   name: ANAYA, HECTOR
job_titles: GENERAL LABORER - DSS
department: DEPARTMENT OF STREETS AND SANITATION
full_or_part_time: F
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 40.0
hourly_rate: 26.28

3. BM25 Score: 4.210
   name: ALTERIO, GEORGE
job_titles: GENERAL LABORER - DSS
department: DEPARTMENT OF STREETS AND SANITATION
full_or_part_time: F
salary_or_hourly: HOURLY
annual_salary: 111814.14905940594
typical_hours: 40.0
hourly_rate: 26.28

 Step 2: Reranking with CrossEncoder
-------------------

In [19]:
query3 = input('Enter query here: ')
results3 = keyword_search_with_reranking(query3, top_k=3, num_candidates=3)

Enter query here:  FIREFIGHTER-EMT


Searching for: 'FIREFIGHTER-EMT'

 Step 1: BM25 retrieval (getting 3 candidates)
--------------------------------------------------------------------------------

Top-3 BM25 results (before reranking):

1. BM25 Score: 2.955
   name: AQUINO, RICARDO
job_titles: FIREFIGHTER-EMT
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 117996.0
typical_hours: 36.40625
hourly_rate: 44.2503125

2. BM25 Score: 2.955
   name: ACUNA, ANIL
job_titles: FIREFIGHTER-EMT
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 117996.0
typical_hours: 36.40625
hourly_rate: 44.2503125

3. BM25 Score: 2.955
   name: ALONSO, JOSE A
job_titles: FIREFIGHTER-EMT
department: CHICAGO FIRE DEPARTMENT
full_or_part_time: F
salary_or_hourly: SALARY
annual_salary: 117996.0
typical_hours: 36.40625
hourly_rate: 44.2503125

 Step 2: Reranking with CrossEncoder
------------------------------------------------------------------------------